# பாடம் 09 - மேட்டாகாக்னிஷன் வடிவமைப்புத்தளம்


## அமைப்பு

இந்த நோட்புக் Microsoft Agent Framework-ஐ பயன்படுத்தி Metacognition வடிவமைப்பை விளக்குகிறது.

**முன்கூறுகள்:**
- சுற்றுச்சூழல் மாறிகளின் மூலம் Azure OpenAI ஏற்பாடு செய்யப்பட்டிருக்க வேண்டும்
- Azure CLI-ல் உள்நுழைந்திருக்க வேண்டும் (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## மேட்டாகாக்னிஷன் என்றால் என்ன?

மேட்டாகாக்னிஷன் என்பது **சிந்திப்பதைப் பற்றிப் சிந்திப்பது**. கைஐ முகவர்கள் (AI agents) குறித்து பேசும்போது, இது பின்வருவன செய்யக்கூடிய முகவர்களை உருவாக்குவதை குறிக்கிறது:

- தங்களின் வெளியீடுகள் மற்றும் காரணங்காட்டும் செயல்முறையை **சுய-பரிசீலனை** செய்தல்
- **பிழைகள் கண்டறிதல்** மற்றும் மௌனமாக தோல்வியடையாமல் நன்றா மீட்குதல்
- தங்கள் பதில்கள் முழுமையானவை மற்றும் உதவிகரமானவையா என்பதை **பரிசீலனை** செய்தல்
- தொடக்க அணுகுமுறை செயல்படாத போது தங்கள் கூற்று முறையை **தழுவி** மாற்றுதல் (உதாரணமாக, தற்காலிக முறையை பயன்படுத்துதல்)

மேட்டாகாக்னிட்டீவ் முகவர் கேள்விகளுக்கு மட்டும் பதிலளிப்பதில்லை — தன் செயல்திறனை கண்காணித்து உடனடி மாற்றம் செய்து கொள்கிறது.


## முதன்மை மற்றும்Backup கருவிகள்

ஒரு பொதுவான மேட்டாவிவேச்சை மாதிரி **fallback strategy** ஆகும். முகவர் முதலில் பிரதான கருவியை முயற்சிக்கிறார்; அது தோல்வியடையும்போது (எடுத்துக்காட்டு, 404 பிழை), முகவர் அந்த தோல்வியை உணர்ந்து வெளிப்படையாகவே ஒருBackup கருவிக்கு மாறுகிறார்.

இது நிஜ உலக அமைப்புகளை போன்றது, அங்கு முதன்மை சேவைகள் கிடைக்காமலிருக்கும், முகவர்கள் தங்களைத் தானே பரிசோதித்து பதிலாக வேறு வழியைத் தேர்ந்தெடுக்க வேண்டும்.

கீழே நாம் இரண்டு விமான_lookup கருவிகளை வரையறுக்கின்றோம்:
- **முதன்மை** — பாரிஸ், டோக்கியோ, மற்றும் பார்சிலோனா ஆகியவற்றை கவர்கிறது
- **Backup** — பெர்லின், சிட்னி, மற்றும் நியூயார்க் நகரத்தை கவர்கிறது


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## பிழை மீட்பு கொண்ட தானாக சுய-பரிசீலனை செய்யும் முகவர்

கீழே உள்ள முகவருக்கு முதலில் பிரதான விமான பணியமைப்பை முயற்சிக்க, தோல்விகளை அடையாளம் காண, பின்னர் வெளிப்படையாக ரிசர்வ் அமைப்புக்கு திரும்புதல் செய்க என்று அறிவுரை வழங்கப்பட்டுள்ளது. ஒவ்வொரு பதிலும் பிறகு அது சிறிய அளவில் சுயம்தானே மதிப்பீடு செய்து, பயனர் கேள்விக்கு முழுமையாக பதில் அளித்ததா என்பதை பார்க்கிறது.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## சுய மதிப்பீடு முறை

மேட்டக்கொறிதன்மையின்另一面 аспект হল **சுய மதிப்பீடு**: ஒரு தனித்த பிரதிநிதி (அல்லது இரண்டாவது முறையில் அதே பிரதிநிதி) பதிலை முழுமை, துல்லியம் மற்றும் உதவிக்கரம் என்பவற்றிற்கு மதிப்பாய்வு செய்கிறார்.

கீழே நாம் `ResponseEvaluator` பிரதிநிதியை உருவாக்குகிறோம், இது பயண பிரதிநிதி பதில்களை மூன்று பரிமாணங்களில் மதிப்பிடுகிறது.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## சுருக்கம்

இந்த பாடத்தில் நீங்கள் Microsoft Agent Framework பயன்படுத்தி **மெட்டாக்னொசிட்டிவ் ஏஜெண்ட்களைக்** கட்டமைப்பது எப்படி என்பதை கற்றுக் கொண்டீர்கள்:

- **சுய-திரும்பிப் பார்வை**: தங்களுடைய காரணம்சார்ந்த செயல்முறைகளை கண்காணித்து என்ன நடந்தது என்பதனை வெளிப்படையாக தொடர்பு கொள்வதற்கான ஏஜெண்ட்கள்.
- **பிழை சீர்திருத்தம் மாற்றுப் முறைகளுடன்**: முதன்மை + பதிலடி கருவி மாதிரியானது, இதில் ஏஜெண்ட் தோல்விகளை (உதா., 404 பிழைகள்) கண்டறிந்து தானாகவே மாற்று மூலத்தை முயற்சிக்கும்.
- **சுய மதிப்பீடு**: பதில்களின் முழுமை, துல்லியம் மற்றும் உதவித்தன்மைக்கு மதிப்பீடு செய்யும் தனித்த ஏஜெண்ட்.

இந்த மாதிரிகள் ஏஜெண்ட்களை அதிகரித்துக் கொண்ட, வெளிப்படையான மற்றும் நம்பிக்கைக்குரியவையாக ஆக்குகின்றன — உற்பத்தி பயன்பாடுகளுக்கான முக்கியமான பண்புகள்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
